# 12 — Data Transformation for Clustering (Objective 1)

**Objective.** Turn the preprocessed dataset into the form a distance-based clustering method needs:
the right rows, the right columns, in a form where each column counts fairly in the distance between two
warehouses.

The structure of the clustering was established in the earlier EDA: segments are defined by the four performance measures (shipment weight, refill requests, transport issues and breakdowns), with all conditions held back for profiling; the 908 unrated warehouses receive their own segment by rule rather than by clustering. This notebook settles the remaining preparation questions — whether anything needs encoding (§2), whether any input's shape should be transformed before scaling (§3), and which scaler gives the four inputs equal weight in the distance calculation (§4).

**Input:** `data/preprocessed/warehouse_preprocessed.csv`
**Output:** `Obj_1_Clustering/data/processed/clustering_base.csv` — the selected rows and inputs, unscaled;
`Obj_1_Clustering/feature_engine/feature_spec.md` — the record of what enters the clustering and why.

The scaler is **chosen** here but **fitted** in the feature engineering step, once the final set of inputs is known. Fitting it here would have to be repeated if a feature is added later.

## 0. Setup

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

df = load_preprocessed()
paths = obj_paths(1)
print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} columns from {PREPROCESSED_FILE.name}")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

---
## 1. Rows and inputs

**Inputs.** The four performance measures that remain once `storage_issue_reported_l3m` moves to
profiling: `product_wg_ton`, `num_refill_req_l3m`, `transport_issue_l1y`, `wh_breakdown_l3m`.

**Rows.** Warehouses with `is_unrated_warehouse == 0`. The 908 unrated warehouses are not discarded: they
receive their own segment by rule in the modelling step, so every one of the 25,000 warehouses ends with a segment.

`Ware_house_ID` is carried alongside as the key — never as an input — so every cluster label can be
joined back to the full warehouse record for profiling.

In [ ]:
inputs = ["product_wg_ton", "num_refill_req_l3m", "transport_issue_l1y", "wh_breakdown_l3m"]
rated = df["is_unrated_warehouse"] == 0

base = df.loc[rated, ["Ware_house_ID"] + inputs].reset_index(drop=True)

print(f"rated warehouses — clustered            : {len(base):,}")
print(f"unrated warehouses — own segment by rule : {int((~rated).sum()):,}")
print(f"total                                    : {len(base) + int((~rated).sum()):,}\n")

assert len(base) + int((~rated).sum()) == len(df) == 25_000
assert base[inputs].notna().all().all()
assert base["Ware_house_ID"].is_unique

base[inputs].describe().T.round(3)

> **Interpretation.**
>
> - 24,092 rated warehouses go to the clustering and 908 are set aside for their rule-based
>   segment — 25,000 in total. No input has a gap, and the key is still unique.
>
> - Setting the unrated warehouses aside changes two of the inputs, exactly as the earlier EDA led us to expect:
>   - **Shipment weight** now starts at **4,055 t** — the lighter shipments all belonged to unrated warehouses — and
>     its mean is 22,730.99 t.
>   - **Breakdowns** now run from **1 to 6**, because every warehouse with zero breakdowns was unrated.
>
> - Refill requests (0 to 8, mean 4.093) and transport issues (0 to 5, median 0, mean 0.779) are essentially unchanged from the full 25,000-warehouse dataset — as the preliminary analysis established, the unrated warehouses did not differ meaningfully on these two measures.

---
## 2. Encoding

Encoding converts text categories into numbers. It is needed only for a column that is (a) an input to the model
and (b) not already numeric.

- **(a)** No condition is an input — every categorical column in this dataset is a condition, and segments are
  defined by performance measures only. The profiles in the evaluation step will describe conditions *as they are*:
  the share of each certificate grade, zone or location type within a segment. Shares and medians need no encoding.
- **(b)** The check below confirms the four inputs are already numeric.

In [ ]:
print(base[inputs].dtypes.to_string())
assert all(pd.api.types.is_numeric_dtype(base[c]) for c in inputs)
print("\nall four inputs are numeric")

> **Interpretation.**
>
> - All four inputs are stored as `int64` — whole-number measures, as the preprocessing step left them.
> - Nothing in the clustering input is text.

> **Decision — no encoding needed.**
>
> - No categorical column is a clustering input — segments are defined by performance measures only.
> - Every clustering input is already numeric.
> - One-hot and ordinal encoding would only be needed if conditions defined the segments.
> - The EDA did not support conditions defining the segments.
> - Certificate-grade order still matters only for presentation in the profiling step, where profiles list grades in business order.

---
## 3. Shape

A transformation such as a logarithm changes the *shape* of a column before it is scaled. It can be worth doing
when a few extreme values would otherwise sit so far from the rest that they pull cluster centres toward
themselves or end up in a cluster of their own.

It also has a cost that matters for segmentation. A log transform makes differences at the low end count for more
than equal differences at the high end — the step from 0 to 1 transport issue would count for more than the step
from 4 to 5. So a transformation is justified only if the extremes are strong enough to distort the result.

The table measures, for the 24,092 rated warehouses:

- **skew and kurtosis** — how lopsided and how heavy-tailed each input is;
- **after standardising** (mean 0, standard deviation 1): the largest distance of any warehouse from the mean, in
  standard deviations, and the share of warehouses more than 3 standard deviations away. A column with many
  warehouses far beyond 3 would be the one extremes could distort.

In [ ]:
z = (base[inputs] - base[inputs].mean()) / base[inputs].std(ddof=0)

shape = pd.DataFrame({
    "min": base[inputs].min(),
    "median": base[inputs].median(),
    "max": base[inputs].max(),
    "skew": base[inputs].skew().round(3),
    "kurtosis": base[inputs].kurtosis().round(3),
    "largest_abs_z": z.abs().max().round(2),
    "pct_beyond_3_sd": (100 * (z.abs() > 3).mean()).round(2),
})
shape

> **Interpretation.**
>
> Three of the four inputs — shipment weight, refill requests and breakdowns — have no warehouse beyond 3 standard deviations, with small skews (0.33, −0.08, 0.07) and the most extreme warehouse no further than 2.86 standard deviations from the mean. Refills and breakdowns have kurtosis around −1.2, meaning they are flat rather than heavy-tailed. Transport issues is the exception: skew 1.60, median 0, and 1.39% of warehouses lie beyond 3 standard deviations. The histograms below show what each distribution looks like in detail.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, c in zip(axes, inputs):
    if base[c].nunique() <= 60:
        sns.histplot(base[c], discrete=True, ax=ax)
    else:
        sns.histplot(base[c], bins=40, ax=ax)
    ax.set_title(c, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("")
fig.suptitle("The four clustering inputs — rated warehouses only", fontweight="bold", y=1.03)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - **Shipment weight, refill requests and breakdowns have no warehouse beyond 3 standard deviations.** Their skews
>   are small (0.331, −0.078, 0.068) and their most extreme warehouses lie 2.86, 1.57 and 1.66 standard deviations
>   from the mean. Refills and breakdowns have kurtosis close to −1.2: flat, not heavy-tailed. Read from the
>   histograms: refills are flat apart from the lower bar at 2; breakdowns are most common at 2 and 3; shipment
>   weight has several peaks, as the preliminary analysis found for all warehouses.
> - **Transport issues is the one lopsided input** — skew 1.599, median 0, and 1.39% of warehouses beyond 3 standard
>   deviations. With a mean of 0.779 and a standard deviation of 1.203, four issues sits 2.68 standard deviations
>   above the mean and five issues 3.51. So that 1.39% is exactly the warehouses reporting **five** transport issues,
>   and 3.51 is as far as any warehouse lies.

> **Decision — no shape transformation.**
>
> - The only input with warehouses beyond 3 standard deviations reaches 3.51 at most — at the edge of the range,
>   not far outside it.
> - Those warehouses are the ones with the most transport issues: a group that matters for performance segments,
>   not a distortion to suppress. A log or square-root transform would compress exactly that end, counting the step
>   from four issues to five as smaller than the step from none to one.
> - The other three inputs show no extremes at all.
>
> Rejected: a log or square-root transformation of `transport_issue_l1y`.

---
## 4. Which scaler

**What "fair weight" means, precisely.** K-Means and Ward's hierarchical method — the two algorithms the planned method
names — both work with **squared Euclidean distance**. For two warehouses picked at random, the expected squared
distance between them is twice the sum of the columns' variances. So **a column's share of the total variance is
exactly its share of the typical distance between warehouses** — its weight in deciding which warehouses end up
together. The EDA found that unscaled, `product_wg_ton` would hold 99.18% of that weight across all numeric
columns.

Three standard scalers, each measured on the same four inputs:

| Scaler | What it does |
|---|---|
| `StandardScaler` | subtracts the mean, divides by the standard deviation |
| `MinMaxScaler` | shifts and divides so each column runs from 0 to 1 |
| `RobustScaler` | subtracts the median, divides by the interquartile range (Q3 − Q1); designed to resist extreme values |

With four inputs, perfectly equal weighting would give each **25%**.

In [ ]:
X = base[inputs]

variances = {"unscaled": X.var(ddof=0)}
for name, scaler in [("StandardScaler", StandardScaler()),
                     ("MinMaxScaler", MinMaxScaler()),
                     ("RobustScaler", RobustScaler())]:
    variances[name] = pd.DataFrame(scaler.fit_transform(X), columns=inputs).var(ddof=0)

weight_pct = (100 * pd.DataFrame(variances) / pd.DataFrame(variances).sum()).round(2)

print("what each scaler divides by:")
print(pd.DataFrame({
    "standard_deviation": X.std(ddof=0).round(3),
    "range_max_minus_min": X.max() - X.min(),
    "interquartile_range": X.quantile(0.75) - X.quantile(0.25),
}).to_string())
print("\neach input's share of the typical squared distance, % (equal weighting = 25%):")
weight_pct

> **Interpretation.**
>
> Even before the bar chart, the table makes the choice clear. StandardScaler gives every input exactly 25% of the distance weight — by construction, since it divides each column by its standard deviation and sets every variance to 1. MinMaxScaler's weights are driven by each column's range: refill requests (0–8) and breakdowns (1–6) produce similar-sized ranges and together absorb 65.75% of the weight, leaving shipment weight with only 15.75%. RobustScaler divides by the interquartile range; transport issues has a narrow IQR of 1 (Q1 = 0, Q3 = 1), so even a small raw difference becomes a large scaled difference, and transport issues alone absorbs 54.66% of the total distance weight.

In [ ]:
plot = weight_pct.drop(columns="unscaled").reset_index(names="input").melt(
    id_vars="input", var_name="scaler", value_name="weight_pct")

plt.figure(figsize=(11, 4.5))
sns.barplot(data=plot, x="scaler", y="weight_pct", hue="input")
plt.axhline(25, color="black", linestyle="--", linewidth=1)
plt.text(2.45, 25.8, "equal weight (25%)", ha="right", fontsize=9)
plt.ylabel("% of typical squared distance"); plt.xlabel("")
plt.title("How much each input would count in the clustering, under each scaler", fontweight="bold")
plt.legend(title="", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - The three scalers give very different answers to *how much does each measure count?*
>
> | Input | StandardScaler | MinMaxScaler | RobustScaler |
> |---|---|---|---|
> | `product_wg_ton` | 25.00% | 15.75% | 18.85% |
> | `num_refill_req_l3m` | 25.00% | 33.92% | 16.04% |
> | `transport_issue_l1y` | 25.00% | 18.49% | **54.66%** |
> | `wh_breakdown_l3m` | 25.00% | 31.83% | 10.45% |
>
> - **StandardScaler** gives every input exactly 25%, by construction: it sets every variance to 1.
> - **MinMaxScaler** would make refills and breakdowns together take 65.75% of the distance weight.
> - **RobustScaler** would make transport issues take 54.66%, more than the other three inputs combined.
> - Unscaled, shipment weight would take 100.00%, as the EDA anticipated.

> **Decision — scaling approach.**
>
> - Adopt `StandardScaler`.
> - Reject no scaling: shipment weight would dominate the distance calculation.
> - Reject `MinMaxScaler`: it gives refills more than twice the weight of shipment weight.
> - Reject `RobustScaler`: it gives transport issues most of the distance weight.
> - Use the same scaled inputs for K-Means and Ward's hierarchical method, since both work with squared Euclidean distance.
> - Fit the scaler in the feature engineering step on the final input set.

---
## 5. The chosen scaler on the four inputs

A preview of what the feature engineering step will produce: the inputs after standard scaling, to confirm they end on a common footing.
This fit is **for inspection only** and is not saved. The feature engineering step fits the scaler on the final set of inputs.

**No train/test split is involved, and none is needed.** Supervised models (Objectives 2 and 3) hold back a test
set to check predictions against known answers. Clustering has no known answers to check: its purpose is to
describe *these* 24,092 warehouses, and the evaluation step uses internal measures computed on the same
warehouses. Fitting the scaler on all of them is therefore not leakage — there is no held-out set for information
to leak into.

In [ ]:
preview = pd.DataFrame(StandardScaler().fit_transform(base[inputs]), columns=inputs)
print(preview.describe().T[["mean", "std", "min", "max"]].round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
sns.boxplot(data=base[inputs], orient="h", ax=axes[0])
axes[0].set_title("Before scaling — recorded units")
sns.boxplot(data=preview, orient="h", ax=axes[1])
axes[1].set_title("After StandardScaler — standard deviations from the mean")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - After standard scaling every input has mean 0 and standard deviation 1, so all four now share
>   one axis.
>
> - Read from the boxplots: **before scaling only shipment weight is visible** — the other three are squashed against
>   zero, because their units are thousands of times smaller. That is the EDA's finding in picture form. **After
>   scaling**, the four boxes have comparable spreads.
>
> - Two sets of points fall beyond the whiskers, and both are expected:
>   - **Transport issues** at the values 3, 4 and 5 — 1.85, 2.68 and 3.51 standard deviations above the mean — because
>     the box is narrow (Q1 = 0, Q3 = 1).
>   - **Shipment weight** at its maximum, 2.86. Among rated warehouses the upper fence is 30,129 + 1.5 × 16,059 =
>     54,217.5 t, and the heaviest shipments (up to 55,151 t) lie just beyond it.
>
> - Nothing lies far outside the rest, consistent with the earlier decision not to transform the inputs. After scaling, the inputs run from −1.646 to 2.858
>   (shipment weight), −1.570 to 1.499 (refills), −0.648 to 3.509 (transport issues) and −1.656 to 1.512
>   (breakdowns).

---
## 6. Save

Two files:

- **`clustering_base.csv`** — `Ware_house_ID` and the four inputs, unscaled, for the 24,092 rated warehouses.
  Unscaled on purpose: the feature engineering step may add features, then fits the scaler on the final set.
- **`feature_spec.md`** — a plain record of the rows, inputs, excluded columns and chosen scaler, with the decision
  behind each. It is written from this notebook's variables, so its counts match what was saved.

In [ ]:
save_table(base, paths["processed"] / "clustering_base.csv", index=False)

spec = f'''# Objective 1 — clustering feature specification

Written by `notebooks/12_data_transformation.ipynb`. Updated by the feature engineering notebook if the final inputs change.

## Rows
- **{len(base):,} rated warehouses** (`is_unrated_warehouse == 0`) are clustered.
- **{int((~rated).sum()):,} unrated warehouses** receive their own segment by rule, not by clustering.

## Inputs (unscaled in `data/processed/clustering_base.csv`)
| Input | Why |
|---|---|
| `product_wg_ton` | shipment volume; kept from the near-duplicate pair |
| `num_refill_req_l3m` | refill activity; unrelated to every other measure (as shown in the EDA) |
| `transport_issue_l1y` | reported transport problems (as defined in the EDA) |
| `wh_breakdown_l3m` | reported breakdowns (as defined in the EDA) |

`Ware_house_ID` is carried as the key only.

## Not inputs
| Column(s) | Why | Used for |
|---|---|---|
| `storage_issue_reported_l3m` | near-duplicate of `product_wg_ton`, r = 0.987 | profiling |
| every condition — numeric, 0/1 and categorical | segments are defined by performance only | profiling |
| `wh_est_year_missing` | describes how data was recorded, not the warehouse | restricting year-based profiles to recorded years |

## Transformation
- **Encoding:** none — every input is numeric; no condition is an input.
- **Shape transform:** none — only `transport_issue_l1y` has warehouses beyond 3 standard deviations
  ({shape.loc["transport_issue_l1y", "pct_beyond_3_sd"]}%, furthest {shape.loc["transport_issue_l1y", "largest_abs_z"]} SD); a log
  transform would compress the high end, which is the part that matters for segmentation.
- **Scaler:** `StandardScaler` — the only candidate giving each input equal weight
  ({weight_pct.loc["product_wg_ton", "StandardScaler"]}% each). `RobustScaler` would give `transport_issue_l1y`
  {weight_pct.loc["transport_issue_l1y", "RobustScaler"]}%; `MinMaxScaler` would give `num_refill_req_l3m`
  {weight_pct.loc["num_refill_req_l3m", "MinMaxScaler"]}% against {weight_pct.loc["product_wg_ton", "MinMaxScaler"]}% for
  `product_wg_ton`. Fitted in the feature engineering step, on the final inputs, over all rated warehouses (no train/test split in clustering).
'''
spec_path = paths["feature_engine"] / "feature_spec.md"
spec_path.write_text(spec, encoding="utf-8")
print(f"saved  {spec_path.relative_to(PROJECT_ROOT)}")

---
## 7. Checks

The saved file is read back and checked against the decisions it is meant to implement.

In [ ]:
back = pd.read_csv(paths["processed"] / "clustering_base.csv")

assert back.shape == (24_092, 5), back.shape
assert list(back.columns) == ["Ware_house_ID"] + inputs
assert back.isna().sum().sum() == 0
assert back["Ware_house_ID"].is_unique
unrated_ids = set(df.loc[df["is_unrated_warehouse"] == 1, "Ware_house_ID"])
assert not unrated_ids & set(back["Ware_house_ID"]), "an unrated warehouse was included"
assert set(back["Ware_house_ID"]) <= set(df["Ware_house_ID"])
assert "storage_issue_reported_l3m" not in back.columns
pd.testing.assert_frame_equal(back, base)

print("all checks passed")
print(f"clustering_base.csv : {back.shape[0]:,} rows x {back.shape[1]} columns, no nulls, no unrated warehouses")

---
## Summary

**Output.**
- `Obj_1_Clustering/data/processed/clustering_base.csv` — 24,092 rated warehouses; `Ware_house_ID` plus the four
  inputs, unscaled; no gaps.
- `Obj_1_Clustering/feature_engine/feature_spec.md` — rows, inputs, excluded columns and transformations, with the
  decision behind each.

Three preparation questions were settled here. No encoding is needed because every clustering input is already numeric and no condition enters the clustering — conditions are used only to profile segments afterwards. No shape transformation is applied because the only input with warehouses beyond 3 standard deviations — transport issues, reaching at most 3.51 — represents the warehouses with the most issues, a group that matters for segmentation rather than a distortion to suppress. StandardScaler is chosen because it is the only candidate giving each of the four inputs equal weight (25% each); MinMaxScaler would give refill requests more than twice the weight of shipment weight, and RobustScaler would give transport issues more than half the total distance weight on its own.

**Why no train/test split.** Clustering has no known answers to test predictions against. It describes these
24,092 warehouses, and the evaluation step uses internal measures on the same warehouses, so fitting the scaler on
all of them leaks nothing.

**Handed to the feature engineering step:** the unscaled base inputs and the choice of StandardScaler. The remaining question is whether any derived feature adds information the four inputs do not already carry; if one is added, the equal-weight reasoning extends to it and `feature_spec.md` is updated.